# 03 — Exploratory Data Analysis

This notebook explores the cleaned movie dataset before machine learning.

Main questions:
- How is the dataset distributed across years and genres?
- What does movie rating/score look like?
- What is the typical movie runtime?
- How are budget and gross revenue distributed?
- Is budget related to gross revenue?
- How are profit and ROI distributed?
- Are runtime, votes, budget, gross and score related?
- Are there useful release-month and company/director patterns?

**Important:** Missing budget and gross values are kept as missing. They are not treated as zero.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

movies = pd.read_csv("../data/processed/movies_cleaned.csv")

# Remove only rows that are completely empty.
movies = movies.dropna(how="all").reset_index(drop=True)

movies["released_date"] = pd.to_datetime(movies["released_date"], errors="coerce")

numeric_columns = ["year", "score", "votes", "budget", "gross", "runtime", "release_year"]
for col in numeric_columns:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

print("Dataset shape:", movies.shape)

In [ ]:
movies.head()

In [ ]:
movies.info()

In [ ]:
movies.describe().T

## Missing values

Budget has a relatively large amount of missing data. This is important because financial analysis must use only records where the required financial values are actually known.

In [ ]:
missing = pd.DataFrame({
    "missing_count": movies.isna().sum(),
    "missing_percentage": (movies.isna().mean() * 100).round(2)
}).sort_values("missing_percentage", ascending=False)

missing

## Movies by year

In [ ]:
movies_per_year = movies["year"].value_counts().sort_index()
movies_per_year

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x=movies_per_year.index.astype(int), y=movies_per_year.values)
plt.title("Number of Movies in the Dataset by Year")
plt.xlabel("Year")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

The dataset contains a different number of sampled movies in different years. Therefore, these counts describe this dataset rather than the total number of movies released by the film industry.

## Average score over time

In [ ]:
average_score_by_year = movies.groupby("year")["score"].mean().sort_index()
average_score_by_year

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(x=average_score_by_year.index, y=average_score_by_year.values, marker="o")
plt.title("Average Movie Score by Year")
plt.xlabel("Year")
plt.ylabel("Average Score")
plt.tight_layout()
plt.show()

## Genre distribution

In [ ]:
genre_counts = movies["genre"].value_counts()
genre_counts

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x=genre_counts.values, y=genre_counts.index)
plt.title("Movie Distribution by Genre")
plt.xlabel("Number of Movies")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

In [ ]:
genre_percentage = movies["genre"].value_counts(normalize=True).mul(100).round(2)
genre_percentage

### Average score by genre

Very small genres should not be interpreted as reliable rankings. We therefore show the full table but also create a second table using a minimum sample size.

In [ ]:
genre_score = movies.groupby("genre")["score"].agg(["mean", "count"]).sort_values("mean", ascending=False)
genre_score

In [ ]:
genre_score_reliable = genre_score[genre_score["count"] >= 50].sort_values("mean", ascending=False)
genre_score_reliable

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=genre_score_reliable.reset_index(), x="mean", y="genre")
plt.title("Average Score by Genre (Genres with at Least 50 Movies)")
plt.xlabel("Average Score")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

## Runtime

In [ ]:
movies["runtime"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(movies["runtime"].dropna(), bins=30, kde=True)
plt.title("Distribution of Movie Runtime")
plt.xlabel("Runtime (minutes)")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

In [ ]:
runtime_bins = pd.cut(
    movies["runtime"],
    bins=[0, 90, 120, 150, 180, 240, 400],
    labels=["Under 90", "90–120", "120–150", "150–180", "180–240", "240+"]
)

runtime_distribution = runtime_bins.value_counts().sort_index()
runtime_distribution

In [ ]:
runtime_distribution.plot(kind="bar", figsize=(10, 6))
plt.title("Movies by Runtime Range")
plt.xlabel("Runtime Range (minutes)")
plt.ylabel("Number of Movies")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
runtime_score_corr = movies["runtime"].corr(movies["score"])
print("Correlation between runtime and score:", round(runtime_score_corr, 4))

runtime_score = movies.groupby(runtime_bins, observed=False)["score"].agg(["mean", "count"])
runtime_score

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=runtime_score.index, y=runtime_score["mean"])
plt.title("Average Movie Score by Runtime Range")
plt.xlabel("Runtime Range")
plt.ylabel("Average Score")
plt.tight_layout()
plt.show()

## Long-runtime movies

In [ ]:
long_movies = movies[movies["runtime"] > 180][
    ["name", "runtime", "year", "genre", "score"]
].sort_values("runtime", ascending=False)

long_movies.head(30)

## Budget

In [ ]:
budget_data = movies[movies["budget"].notna()].copy()

print("Movies with known budget:", len(budget_data))
print("Movies with missing budget:", movies["budget"].isna().sum())
budget_data["budget"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(budget_data["budget"], bins=40, kde=True)
plt.title("Distribution of Movie Budgets")
plt.xlabel("Budget")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

In [ ]:
budget_data["log_budget"] = np.log1p(budget_data["budget"])

plt.figure(figsize=(10, 6))
sns.histplot(budget_data["log_budget"], bins=40, kde=True)
plt.title("Log-Transformed Movie Budget Distribution")
plt.xlabel("log(1 + Budget)")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

In [ ]:
highest_budget = budget_data[
    ["name", "budget", "year", "genre", "score"]
].sort_values("budget", ascending=False)

highest_budget.head(20)

## Gross revenue

In [ ]:
gross_data = movies[movies["gross"].notna()].copy()

print("Movies with known gross revenue:", len(gross_data))
print("Movies with missing gross revenue:", movies["gross"].isna().sum())
gross_data["gross"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(gross_data["gross"], bins=40, kde=True)
plt.title("Distribution of Movie Gross Revenue")
plt.xlabel("Gross Revenue")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

In [ ]:
gross_data["log_gross"] = np.log1p(gross_data["gross"])

plt.figure(figsize=(10, 6))
sns.histplot(gross_data["log_gross"], bins=40, kde=True)
plt.title("Log-Transformed Gross Revenue Distribution")
plt.xlabel("log(1 + Gross Revenue)")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

## Budget vs gross revenue

In [ ]:
budget_gross = movies[movies["budget"].notna() & movies["gross"].notna()].copy()

print("Movies with both budget and gross:", len(budget_gross))
print("Correlation:", round(budget_gross["budget"].corr(budget_gross["gross"]), 4))

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=budget_gross, x="budget", y="gross", alpha=0.45)
plt.title("Movie Budget vs Gross Revenue")
plt.xlabel("Budget")
plt.ylabel("Gross Revenue")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=budget_gross,
    x=np.log1p(budget_gross["budget"]),
    y=np.log1p(budget_gross["gross"]),
    alpha=0.45
)
plt.title("Log Budget vs Log Gross Revenue")
plt.xlabel("log(1 + Budget)")
plt.ylabel("log(1 + Gross Revenue)")
plt.tight_layout()
plt.show()

## Profit and ROI

In [ ]:
financial = movies[movies["budget"].notna() & movies["gross"].notna() & (movies["budget"] > 0)].copy()

financial["profit"] = financial["gross"] - financial["budget"]
financial["roi"] = financial["profit"] / financial["budget"]
financial["roi_percent"] = financial["roi"] * 100

print("Movies with both budget and gross:", len(financial))
financial[["profit", "roi_percent"]].describe()

In [ ]:
top_profit = financial[
    ["name", "budget", "gross", "profit", "roi_percent", "genre", "score"]
].sort_values("profit", ascending=False)

top_profit.head(20)

In [ ]:
top_roi = financial[
    ["name", "budget", "gross", "profit", "roi_percent", "genre", "score"]
].sort_values("roi_percent", ascending=False)

top_roi.head(20)

ROI can be extreme for very low-budget movies. For a more stable comparison, inspect ROI for movies with a minimum budget threshold as well.

In [ ]:
roi_reliable = financial[financial["budget"] >= 1_000_000].copy()

roi_reliable[
    ["name", "budget", "gross", "profit", "roi_percent", "genre", "score"]
].sort_values("roi_percent", ascending=False).head(20)

## Votes vs score

In [ ]:
print("Correlation between votes and score:", round(movies["votes"].corr(movies["score"]), 4))

plt.figure(figsize=(10, 6))
sns.scatterplot(data=movies, x="votes", y="score", alpha=0.4)
plt.xscale("log")
plt.title("Votes vs Movie Score")
plt.xlabel("Votes (log scale)")
plt.ylabel("Score")
plt.tight_layout()
plt.show()

## Correlation matrix

In [ ]:
correlation_columns = [
    "year", "score", "votes", "budget",
    "gross", "runtime", "release_year"
]

correlation_matrix = movies[correlation_columns].corr()
correlation_matrix

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## Release month

In [ ]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

release_month_counts = (
    movies["release_month"]
    .value_counts()
    .reindex(month_order)
)

release_month_counts

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x=release_month_counts.index, y=release_month_counts.values)
plt.title("Movies by Release Month")
plt.xlabel("Release Month")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Top directors and production companies

In [ ]:
top_directors = movies["director"].value_counts().head(15)
top_companies = movies["company"].value_counts().head(15)

display(top_directors)
display(top_companies)

In [ ]:
director_score = (
    movies.groupby("director")["score"]
    .agg(["mean", "count"])
    .query("count >= 10")
    .sort_values("mean", ascending=False)
)

director_score.head(15)

## EDA summary

At the end of this notebook, record the main findings from your actual outputs. Avoid treating dataset patterns as universal movie-industry facts, because the dataset covers a limited period and appears to contain sampled/capped yearly records.